In [ ]:
!pip install yfinance pandas numpy torch scikit-learn matplotlib -q

In [ ]:
import os, numpy as np, pandas as pd, yfinance as yf, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings("ignore")

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)


In [ ]:
# download SPX and VIX data
path = "data/raw/spx_daily.csv"
if os.path.exists(path):
    spx = pd.read_csv(path, index_col="Date", parse_dates=True)
else:
    spx = yf.download("^GSPC", start="2005-01-01", end="2025-12-31", auto_adjust=True)
    spx.to_csv(path)

path = "data/raw/vix_term_structure.csv"
if os.path.exists(path):
    vix_raw = pd.read_csv(path, index_col="Date", parse_dates=True)
else:
    frames = {}
    for label, ticker in {"VIX": "^VIX", "VIX3M": "^VIX3M", "VIX6M": "^VIX6M"}.items():
        try:
            df = yf.download(ticker, start="2005-01-01", end="2025-12-31", auto_adjust=True)
            if len(df) > 0: frames[label] = df["Close"].squeeze()
        except: pass
    vix_raw = pd.DataFrame(frames)
    vix_raw.index.name = "Date"
    vix_raw.to_csv(path)

print(f"SPX: {len(spx)} days, VIX: {len(vix_raw)} days")


In [ ]:
# build the 12 features the autoencoder will use
close = spx["Close"].squeeze()
high = spx["High"].squeeze()
low = spx["Low"].squeeze()
log_ret = np.log(close / close.shift(1))
squared = log_ret ** 2

feat = pd.DataFrame(index=close.index)

# realized vol
for h in [1, 5, 22]:
    feat[f"RV_{h}d"] = np.sqrt(squared.rolling(h).sum() * (252 / h))

# parkinson vol
feat["parkinson_vol"] = np.sqrt((1 / (4 * np.log(2))) * (np.log(high / low) ** 2)) * np.sqrt(252)

# VIX features
feat["vix_level"] = vix_raw.get("VIX")
if "VIX" in vix_raw.columns and "VIX3M" in vix_raw.columns:
    feat["slope_3m_spot"] = vix_raw["VIX3M"] - vix_raw["VIX"]
if "VIX3M" in vix_raw.columns and "VIX6M" in vix_raw.columns:
    feat["slope_6m_3m"] = vix_raw["VIX6M"] - vix_raw["VIX3M"]
if all(c in vix_raw.columns for c in ["VIX", "VIX3M", "VIX6M"]):
    feat["curvature"] = vix_raw["VIX"] - 2 * vix_raw["VIX3M"] + vix_raw["VIX6M"]
if "VIX" in vix_raw.columns and "VIX3M" in vix_raw.columns:
    feat["contango_flag"] = (vix_raw["VIX3M"] > vix_raw["VIX"]).astype(int)
if "VIX" in vix_raw.columns:
    feat["vix_1d_change"] = vix_raw["VIX"].pct_change()
    feat["vix_zscore_20d"] = (
        (vix_raw["VIX"] - vix_raw["VIX"].rolling(20).mean()) / vix_raw["VIX"].rolling(20).std()
    )
    feat["vix_ma_ratio"] = vix_raw["VIX"] / vix_raw["VIX"].rolling(60).mean()

feat = feat.dropna()
ae_cols = [c for c in feat.columns]
print(f"Features ({len(ae_cols)}): {ae_cols}")
print(f"Rows: {len(feat)}")


In [ ]:
# also build targets so we can color the latent space plots
fwd_rv = []
sq = squared.values
idx_map = {d: i for i, d in enumerate(close.index)}
for d in feat.index:
    i = idx_map[d]
    end = i + 1 + 22
    if end > len(sq):
        fwd_rv.append(np.nan)
    else:
        fwd_rv.append(np.sqrt(np.sum(sq[i+1:end]) * (252 / 22)))

feat["fwd_rv_22d"] = fwd_rv
feat = feat.dropna()
threshold = np.percentile(feat["fwd_rv_22d"], 90)
feat["spike_label"] = (feat["fwd_rv_22d"] > threshold).astype(int)
ae_cols = [c for c in feat.columns if c not in ["fwd_rv_22d", "spike_label"]]
print(f"Final rows: {len(feat)}, features for AE: {len(ae_cols)}")


In [ ]:
# sliding window dataset
class SeqDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
    def __len__(self):
        return len(self.data) - self.seq_len + 1
    def __getitem__(self, idx):
        return torch.FloatTensor(self.data[idx:idx + self.seq_len])

# autoencoder model
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, input_dim)
        self.input_dim = input_dim

    def encode(self, x):
        _, (h, _) = self.encoder(x)
        return self.to_latent(h.squeeze(0))

    def decode(self, z, seq_len):
        h0 = self.to_hidden(z).unsqueeze(0)
        c0 = torch.zeros_like(h0)
        dec_in = torch.zeros(z.size(0), seq_len, self.input_dim).to(z.device)
        out, _ = self.decoder(dec_in, (h0, c0))
        return self.output(out)

    def forward(self, x):
        z = self.encode(x)
        recon = self.decode(z, x.size(1))
        return recon, z


In [ ]:
# scale and prepare training data (pre-2020 only)
SEQ_LEN = 30
LATENT_DIM = 8
HIDDEN_DIM = 32

scaler = StandardScaler()
train_raw = feat.loc[feat.index < "2020-01-01", ae_cols]
scaler.fit(train_raw)

train_scaled = scaler.transform(train_raw)
all_scaled = scaler.transform(feat[ae_cols])

train_ds = SeqDataset(train_scaled, SEQ_LEN)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
print(f"Training sequences: {len(train_ds)}")


In [ ]:
# train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMAutoencoder(len(ae_cols), HIDDEN_DIM, LATENT_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

losses = []
for epoch in range(50):
    model.train()
    total = 0
    for batch in train_loader:
        batch = batch.to(device)
        recon, z = model(batch)
        loss = nn.MSELoss()(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item() * batch.size(0)
    avg = total / len(train_ds)
    losses.append(avg)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/50  Loss: {avg:.6f}")

plt.plot(losses)
plt.xlabel("Epoch"); plt.ylabel("MSE Loss"); plt.title("Training Loss")
plt.show()


In [ ]:
# extract latent vectors and reconstruction errors
model.eval()
all_ds = SeqDataset(all_scaled, SEQ_LEN)
all_loader = DataLoader(all_ds, batch_size=256, shuffle=False)

latents, errors = [], []
with torch.no_grad():
    for batch in all_loader:
        batch = batch.to(device)
        recon, z = model(batch)
        latents.append(z.cpu().numpy())
        err = ((recon - batch) ** 2).mean(dim=(1, 2)).cpu().numpy()
        errors.append(err)

latents = np.concatenate(latents)
errors = np.concatenate(errors)

# each vector corresponds to the last day of its window
dates = feat.index[SEQ_LEN - 1:][:len(latents)]
latent_df = pd.DataFrame(latents, index=dates, columns=[f"latent_{i}" for i in range(LATENT_DIM)])
latent_df["recon_error"] = errors

# trajectory features
coords = latent_df[[f"latent_{i}" for i in range(LATENT_DIM)]]
velocity = np.sqrt((coords.diff() ** 2).sum(axis=1))
latent_df["latent_velocity"] = velocity
latent_df["latent_acceleration"] = velocity.diff()

crisis_mask = latent_df.index.year.isin([2008, 2020])
if crisis_mask.sum() > 0:
    centroid = coords[crisis_mask].mean().values
    latent_df["crisis_distance"] = np.sqrt(((coords.values - centroid) ** 2).sum(axis=1))

print(f"Latent features: {latent_df.shape}")
print(f"Columns: {list(latent_df.columns)}")


In [ ]:
# t-SNE visualization
aligned_rv = feat.loc[latent_df.index, "fwd_rv_22d"]
aligned_spike = feat.loc[latent_df.index, "spike_label"]

print("Running t-SNE...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
proj = tsne.fit_transform(coords.values)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sc = axes[0].scatter(proj[:, 0], proj[:, 1], c=aligned_rv.values, cmap="YlOrRd", s=3, alpha=0.6)
plt.colorbar(sc, ax=axes[0], label="Forward RV")
axes[0].set_title("Colored by Volatility")

calm = proj[aligned_spike.values == 0]
spike = proj[aligned_spike.values == 1]
axes[1].scatter(calm[:, 0], calm[:, 1], c="steelblue", s=3, alpha=0.4, label="Calm")
axes[1].scatter(spike[:, 0], spike[:, 1], c="red", s=8, alpha=0.7, label="Spike")
axes[1].set_title("Calm vs Spike")
axes[1].legend()

sc = axes[2].scatter(proj[:, 0], proj[:, 1], c=latent_df["recon_error"].values, cmap="plasma", s=3, alpha=0.6)
plt.colorbar(sc, ax=axes[2], label="Recon Error")
axes[2].set_title("Reconstruction Error")

plt.suptitle("Latent Space (t-SNE)")
plt.tight_layout()
plt.show()


In [ ]:
# COVID trajectory
mask = (latent_df.index >= "2019-11-01") & (latent_df.index <= "2020-06-01")
plot_data = latent_df.loc[mask]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(plot_data.index, plot_data["recon_error"], color="purple")
axes[0].axvline(pd.Timestamp("2020-02-19"), color="red", linestyle="--", label="COVID start")
axes[0].set_title("Reconstruction Error"); axes[0].legend()

axes[1].plot(plot_data.index, plot_data["latent_velocity"], color="darkorange")
axes[1].axvline(pd.Timestamp("2020-02-19"), color="red", linestyle="--", label="COVID start")
axes[1].set_title("Latent Velocity"); axes[1].legend()

plt.suptitle("Crisis Trajectory")
plt.tight_layout()
plt.show()


In [ ]:
# save
latent_df.to_csv("data/processed/p1_latent_features.csv")
print("Saved p1_latent_features.csv")
